# Final Preprocessing

This notebook documents the final preprocessing schema used by the saved LightGBM pipeline. The reusable implementation lives in `src/preprocessing.py` as `ICUPreprocessor`.

The final preprocessing decisions were updated after the model experiment in `notebooks/09_model_experiment.ipynb`. The final model removes ID/location columns and leakage probability columns, keeps numeric values unscaled, and produces a 379-feature schema.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path('..')
sys.path.append(str(ROOT))

from src.preprocessing import ICUPreprocessor

raw = pd.read_csv(ROOT / 'data/raw/training_v2.csv')
raw.shape

(91713, 186)

## Split

The same stratified split is used throughout the project:

- `test_size = 0.2`
- `random_state = 42`
- `stratify = hospital_death`

The preprocessor is fit only on the training split.

In [2]:
raw_train, raw_test = train_test_split(
    raw,
    test_size=0.2,
    random_state=42,
    stratify=raw['hospital_death'],
)

y_train = raw_train['hospital_death'].astype(int)
y_test = raw_test['hospital_death'].astype(int)

print('raw_train:', raw_train.shape)
print('raw_test :', raw_test.shape)
print('train death rate:', round(y_train.mean(), 4))
print('test death rate :', round(y_test.mean(), 4))

raw_train: (73370, 186)
raw_test : (18343, 186)
train death rate: 0.0863
test death rate : 0.0863


## Final preprocessing decisions

The final schema removes:

- `encounter_id`
- `patient_id`
- `hospital_id`
- `icu_id`
- `apache_4a_hospital_death_prob`
- `apache_4a_icu_death_prob`

Numeric features are median-imputed with missingness indicators. Binary features are ordinal-encoded. Categorical features are one-hot encoded with infrequent-category handling. Numeric scaling is omitted because the final model is tree-based and evidence packets should preserve readable clinical values.

In [3]:
preprocessor = ICUPreprocessor()
X_train = preprocessor.fit_transform(raw_train)
X_test = preprocessor.transform(raw_test)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('features:', len(preprocessor.feature_names_))
print('numeric columns:', len(preprocessor.numeric_cols_))
print('binary columns:', len(preprocessor.binary_cols_))
print('categorical columns:', len(preprocessor.categorical_cols_))
print('missing indicators:', len(preprocessor.missing_indicator_cols_))

X_train: (73370, 379)
X_test : (18343, 379)
features: 379
numeric columns: 157
binary columns: 15
categorical columns: 7
missing indicators: 155


In [4]:
dropped_cols = preprocessor.id_cols + preprocessor.leakage_cols + [preprocessor.target_col]
leaked_columns_present = [column for column in dropped_cols if column in X_train.columns]

print('Dropped columns present after preprocessing:', leaked_columns_present)
print('Train/test columns match:', list(X_train.columns) == list(X_test.columns))

X_train.head(3)

Dropped columns present after preprocessing: []
Train/test columns match: True


,age,bmi,height,pre_icu_los_days,readmission_status,weight,albumin_apache,apache_2_diagnosis,apache_3j_diagnosis,bilirubin_apache,...,apache_2_bodysystem_Cardiovascular,apache_2_bodysystem_Gastrointestinal,apache_2_bodysystem_Haematologic,apache_2_bodysystem_Metabolic,apache_2_bodysystem_Neurologic,apache_2_bodysystem_Renal/Genitourinary,apache_2_bodysystem_Respiratory,apache_2_bodysystem_Trauma,apache_2_bodysystem_Undefined diagnoses,apache_2_bodysystem_infrequent_sklearn
59226,72.0,32.041330,154.9,0.061111,0.0,76.88,2.0,113.0,501.02,2.40,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81334,58.0,34.334291,190.5,0.032639,0.0,124.60,2.5,113.0,501.01,1.20,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
67173,78.0,32.764004,165.0,2.527083,0.0,89.20,2.9,218.0,1505.01,0.62,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Save processed matrices

The processed matrices are saved for notebooks and report refresh scripts. The final deployable pipeline uses the saved preprocessor artifact, but these CSV files keep exploratory notebooks reproducible.

In [5]:
processed_dir = ROOT / 'data/processed'
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / 'X_train.csv', index=False)
X_test.to_csv(processed_dir / 'X_test.csv', index=False)
y_train.to_csv(processed_dir / 'y_train.csv', index=False)
y_test.to_csv(processed_dir / 'y_test.csv', index=False)
pd.DataFrame(preprocessor.feature_names_).to_csv(
    processed_dir / 'feature_names.csv',
    index=False,
    header=False,
)

print('Saved processed data to', processed_dir)
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('features:', len(preprocessor.feature_names_))

Saved processed data to ../data/processed
X_train: (73370, 379)
X_test : (18343, 379)
features: 379


## Summary

The final preprocessing schema produces 379 model features and removes ID/location/leakage columns before model fitting. This is the schema used by the saved preprocessor artifact, final LightGBM model, SHAP reports, evidence packets, LLM outputs, validation audit, and dashboard.